In [2]:
import concurrent.futures
from datetime import datetime, timezone
from urllib.parse import parse_qs, urlparse
import pandas as pd
import requests


def es_link_valido(url: str, timeout: int = 5) -> bool:
    """Verifica si una URL es estructuralmente válida, no ha expirado

    y responde con un código HTTP 200 OK.
    """
    if (
        pd.isna(url)
        or not isinstance(url, str)
        or not url.startswith(("http://", "https://"))
    ):
        return False

    # 1. Validación offline rápida para firmas de Google Cloud Storage (X-Goog-Expires)
    try:
        parsed = urlparse(url)
        params = parse_qs(parsed.query)
        if "X-Goog-Date" in params and "X-Goog-Expires" in params:
            date_str = params["X-Goog-Date"][0]
            expires_sec = int(params["X-Goog-Expires"][0])
            created_at = datetime.strptime(
                date_str, "%Y%m%dT%H%M%SZ"
            ).replace(tzinfo=timezone.utc)
            # Si el token de la URL ya sobrepasó su tiempo de vida útil
            if datetime.now(timezone.utc).timestamp() > (
                created_at.timestamp() + expires_sec
            ):
                return False
    except Exception:
        pass

    # 2. Validación de disponibilidad y descarga vía HTTP
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
    }
    try:
        # Intento mediante petición HEAD (rápida, solo consulta cabeceras)
        response = requests.head(
            url, timeout=timeout, allow_redirects=True, headers=headers
        )
        if response.status_code == 200:
            return True
        elif response.status_code in (401, 403, 404, 410):
            return False

        # Si el servidor no soporta HEAD (405 Method Not Allowed), intenta un GET ligero
        response = requests.get(
            url, timeout=timeout, stream=True, headers=headers
        )
        return response.status_code == 200
    except Exception:
        return False


def filtrar_csv_por_links(
    archivo_entrada: str = "data.csv",
    archivo_salida: str = "data_filtrado.csv",
    columna_link: str = "link_imagen",
    max_trabajadores: int = 30,
) -> pd.DataFrame:
    # Cargar CSV original
    df = pd.read_csv(archivo_entrada)

    urls = df[columna_link].tolist()
    print(
        f"Validando {len(urls)} enlaces de la columna '{columna_link}' en paralelo..."
    )

    # Procesar verificación multihilo para mayor rapidez
    with concurrent.futures.ThreadPoolExecutor(
        max_workers=max_trabajadores
    ) as executor:
        mascara_validos = list(
            executor.map(lambda u: es_link_valido(u), urls)
        )

    # Filtrar manteniendo el esquema y los tipos de datos originales
    df_filtrado = df[mascara_validos].copy()

    print(f"Registros originales: {len(df)}")
    print(f"Registros válidos/descargables: {len(df_filtrado)}")
    print(
        f"Registros descartados (vencidos/inválidos): {len(df) - len(df_filtrado)}"
    )

    # Guardar manteniendo exactamente la misma estructura de columnas
    df_filtrado.to_csv(archivo_salida, index=False)
    return df_filtrado


if __name__ == "__main__":
    # Ejecución principal
    df_resultado = filtrar_csv_por_links(
        archivo_entrada="data.csv",
        archivo_salida="data_filtrado.csv",
        columna_link="link_imagen",
        max_trabajadores=30,  # Ajustar concurrencia según ancho de banda
    )

Validando 6443 enlaces de la columna 'link_imagen' en paralelo...
Registros originales: 6443
Registros válidos/descargables: 3285
Registros descartados (vencidos/inválidos): 3158


In [4]:
import os
import requests
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm


def _descargar_un_archivo(args):
    """Función auxiliar ejecutada por cada hilo."""
    url, nombre_base, carpeta_destino, extension_defecto = args

    # Encabezados de petición
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
    }

    # 1. Verificar si el archivo ya existe con alguna extensión común para evitar re-descargas
    for ext_test in ['.jpg', '.png', '.webp', '.pdf', extension_defecto]:
        posible_ruta = os.path.join(carpeta_destino, f"{nombre_base}{ext_test}")
        if os.path.exists(posible_ruta) and os.path.getsize(posible_ruta) > 0:
            return "omitido"

    try:
        # Petición GET con timeout
        response = requests.get(url, headers=headers, stream=True, timeout=15)
        response.raise_for_status()

        # Deducir extensión según Content-Type
        content_type = response.headers.get('Content-Type', '').lower()
        if 'image/jpeg' in content_type or 'image/jpg' in content_type:
            ext = '.jpg'
        elif 'image/png' in content_type:
            ext = '.png'
        elif 'image/webp' in content_type:
            ext = '.webp'
        elif 'application/pdf' in content_type:
            ext = '.pdf'
        else:
            ext = extension_defecto

        ruta_guardado = os.path.join(carpeta_destino, f"{nombre_base}{ext}")

        # Guardar en disco con buffer de 32 KB
        with open(ruta_guardado, 'wb') as file:
            for chunk in response.iter_content(chunk_size=32768):
                if chunk:
                    file.write(chunk)

        return "exito"

    except Exception:
        return "error"


def descargar_archivos_paralelo(
    df,
    columna_link='link_imagen',
    columna_nombre='id',
    carpeta_destino='descargas',
    extension_defecto='.jpg',
    max_workers=20  # Número de descargas simultáneas
):
    """
    Descarga archivos en paralelo usando múltiples hilos.
    """
    os.makedirs(carpeta_destino, exist_ok=True)

    # Filtrar URLs válidas
    df_valid = df.dropna(subset=[columna_link]).copy()
    print(f"Preparando la descarga paralela de {len(df_valid)} archivos con {max_workers} hilos...")

    # Preparar lista de tareas
    tareas = []
    for idx, row in df_valid.iterrows():
        url = str(row[columna_link]).strip()

        if columna_nombre and columna_nombre in row and pd.notna(row[columna_nombre]):
            nombre_base = str(row[columna_nombre]).strip()
        else:
            nombre_base = f"archivo_{idx}"

        tareas.append((url, nombre_base, carpeta_destino, extension_defecto))

    # Contadores
    exitosas = 0
    fallidas = 0
    omitidas = 0

    # Ejecutar descargas en paralelo
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = [executor.submit(_descargar_un_archivo, t) for t in tareas]

        for future in tqdm(as_completed(futures), total=len(futures), desc="Descargando en paralelo"):
            resultado = future.result()
            if resultado == "exito":
                exitosas += 1
            elif resultado == "omitido":
                omitidas += 1
            else:
                fallidas += 1

    print(f"\nProceso finalizado:")
    print(f" - Descargadas con éxito: {exitosas}")
    print(f" - Ya existían (omitidas): {omitidas}")
    print(f" - Errores: {fallidas}")
    print(f" - Carpeta: '{os.path.abspath(carpeta_destino)}'")


if __name__ == "__main__":
    archivo_csv = "data_filtrado.csv"
    df = pd.read_csv(archivo_csv)

    descargar_archivos_paralelo(
        df=df,
        columna_link='link_imagen',
        columna_nombre='id',
        carpeta_destino='imagenes_tiendas',
        max_workers=20  # Puedes subir a 30 u 80 si tu conexión/servidor lo soporta
    )

Preparando la descarga paralela de 3285 archivos con 20 hilos...


Descargando en paralelo: 100%|██████████| 3285/3285 [15:34<00:00,  3.52it/s] 


Proceso finalizado:
 - Descargadas con éxito: 2900
 - Ya existían (omitidas): 345
 - Errores: 40
 - Carpeta: 'C:\Users\luigu\OneDrive\Escritorio\Magnum_app\Analisis\imagenes_tiendas'


In [5]:
import pandas as pd

# 1. Cargar el archivo CSV
df = pd.read_csv('data_filtrado_2.csv')

# 2. Contar la frecuencia absoluta de cada categoría
frecuencias = df['tipo_congelador'].value_counts(dropna=False)

# 3. Contar la frecuencia relativa (porcentaje)
porcentajes = (df['tipo_congelador'].value_counts(dropna=False, normalize=True) * 100).round(2)

# 4. Crear un DataFrame resumen
resumen = pd.DataFrame({
    'Frecuencia': frecuencias,
    'Porcentaje (%)': porcentajes
})

print(resumen)

FileNotFoundError: [Errno 2] No such file or directory: 'data_filtrado_2.csv'